# Observation des lignes de courant de vent — large300 (Méso-NH RCE)

**Objectif du stage (rappel).** On a montré que paramétriser le flux de Reynolds $\rho_0\langle u'w'\rangle$ par le seul flux de masse (bulk-plume) est insuffisant : le résidu sous-maille $T_2$ est irréductible ($\sim 44\%$) et structurellement corrélé au flux total. 

**Nouvelle piste.** Regarder directement la *structure spatiale* de l'écoulement via les **lignes de courant** dans le plan $(x,z)$, coupe moyennée sur $y$, et la **confronter à la corrélation** $u'w'$.

Fil rouge : *l'œil voit une organisation cohérente (cellule d'overturning, panaches, rouleaux) que le moment d'ordre 2 $\langle u'w'\rangle$ « voit » mal*. C'est précisément pourquoi une fermeture flux-de-masse échoue, et ce qui motive d'aller chercher la structure (ondelettes, tourbillons).

---
### Trame (d'après les notes)
1. **Calcul de la fonction de courant** $\psi$ (isolignes) — convention de masse anélastique.
2. **Coupe verticale moyennée sur $y$** à $t$ donné (et moyenne temporelle).
3. **Comment voit-on la faible corrélation $u'w'$ ?** — superposer streamlines et carte de $u'w'$.
4. **Que tirer de la visualisation ?** — diagnostics quantitatifs.
5. **Lien avec la paramétrisation du flux de Reynolds.**
6. **Décomposition du courant en ondelettes / Fourier** — échelles dominantes.
7. **Extension aux lignes de courant 3D.**
8. **Isolation des tourbillons** (critère $\lambda_2$ / vorticité / Okubo-Weiss).
9. **Autres idées** (spectres co-spectraux $u'w'$, angle de phase, quadrant analysis...).

# 📖 Les lignes de courant, pas à pas (pour qui découvre)

Ce bloc explique **ce qu'est une ligne de courant** et **comment on la calcule**, en repartant de zéro. Si tu connais déjà, saute directement à la section 0.

---
## Étape 1 — C'est quoi une ligne de courant ?

À un instant donné, en chaque point de l'espace, le vent est un **vecteur** $\vec{V}=(u,w)$ (ici dans le plan vertical $(x,z)$). Une **ligne de courant** (*streamline*) est une courbe qui est, partout, **tangente à ce vecteur vitesse**.

Image intuitive : si tu lâchais une infinité de petites particules sans masse et que tu prenais une photo instantanée des directions qu'elles suivent, les lignes de courant sont les courbes qui « suivent les flèches » du champ de vent. Là où les lignes se resserrent, l'écoulement est rapide ; là où elles s'écartent, il est lent.

⚠️ **Attention à ne pas confondre** trois objets proches :
- **Ligne de courant** (*streamline*) : tangente au champ **à un instant figé**. C'est ce qu'on calcule ici.
- **Trajectoire** (*pathline*) : le chemin réellement parcouru par **une** particule au cours du temps.
- **Ligne d'émission** (*streakline*) : la position de toutes les particules passées par un même point (comme un filet de fumée).

Dans un écoulement **stationnaire** (qui ne change pas dans le temps) les trois coïncident. Sinon elles diffèrent. Nous travaillons sur des **instantanés** (snapshots) → ce sont bien des *streamlines*.

---
## Étape 2 — L'idée clé : la fonction de courant $\psi$

Tracer des lignes tangentes au champ « à la main » serait pénible. Il existe un **raccourci mathématique** très puissant, à une condition.

**La condition : un écoulement incompressible (sans création ni perte de matière).** En 2D, cela s'écrit « la divergence est nulle » :
$$\frac{\partial u}{\partial x} + \frac{\partial w}{\partial z} = 0.$$
Intuitivement : ce qui entre dans une petite boîte en ressort — pas de source ni de puits.

**Le théorème.** Quand cette condition est vraie, il existe une **fonction scalaire** $\psi(x,z)$ — un simple nombre attaché à chaque point — telle que les deux composantes de la vitesse sont ses dérivées :
$$u = -\frac{\partial \psi}{\partial z}, \qquad w = +\frac{\partial \psi}{\partial x}.$$
Cette $\psi$ s'appelle la **fonction de courant**.

**Pourquoi c'est magique ?** On peut vérifier que le long d'une courbe où $\psi$ reste constant, le déplacement est exactement tangent à $\vec V$. Autrement dit :
$$\boxed{\text{les lignes de courant} = \text{les lignes de niveau (isolignes) de } \psi.}$$
Tracer les lignes de courant revient donc à tracer un simple **contour** de $\psi$ (comme les courbes de niveau d'une carte topographique). Un `plt.contour(psi)` suffit.

Bonus : la **différence de $\psi$** entre deux lignes donne le **débit** qui passe entre elles. Resserrement des isolignes ⇔ écoulement intense.

---
## Étape 3 — Le cas de l'atmosphère : fonction de courant **de masse**

L'air n'est pas incompressible : sa densité $\rho_0(z)$ **chute avec l'altitude** (l'air est plus dense en bas). La divergence de la vitesse n'est donc pas nulle.

**La bonne quantité conservée** n'est pas le volume mais la **masse**. L'approximation *anélastique* (standard en convection) dit que c'est le **flux de masse** $(\rho_0 u,\ \rho_0 w)$ qui est sans divergence :
$$\frac{\partial (\rho_0\, u)}{\partial x} + \frac{\partial (\rho_0\, w)}{\partial z} = 0.$$
On définit alors une **fonction de courant de masse** $\psi$ par :
$$\rho_0\, u = -\frac{\partial \psi}{\partial z}, \qquad \rho_0\, w = +\frac{\partial \psi}{\partial x}.$$
Ses isolignes sont les lignes de courant du **transport de masse** — exactement ce qui décrit la circulation de retournement (*overturning*) de la convection.

---
## Étape 4 — Pourquoi pas une simple intégration ? (le piège)

Naïvement, on pourrait obtenir $\psi$ en **intégrant** : par exemple, partir de $w = \frac{1}{\rho_0}\partial_x\psi$ et cumuler $\psi(x,z) = \int_0^x \rho_0\, w\,dx'$.

**Le problème :** cette recette suppose que la condition de divergence nulle est *parfaitement* vérifiée. Or nos données réelles ne le sont jamais exactement : on a moyenné sur un $y$ de taille finie, il y a du bruit turbulent, et la grille est discrète. Du coup :
- intégrer **horizontalement** ($\int \rho_0 w\, dx$) ou **verticalement** ($-\int \rho_0 u\, dz$) donne **deux résultats différents** ;
- le résultat **dépend du chemin** suivi pour intégrer. Ce n'est pas acceptable.

---
## Étape 5 — La solution propre : décomposition de Helmholtz + équation de Poisson

Tout champ de vecteurs se décompose de façon unique (théorème de **Helmholtz**) en deux morceaux :
$$\text{flux de masse} = \underbrace{\text{partie rotationnelle}}_{\text{tourne, porte la circulation}} + \underbrace{\text{partie divergente}}_{\text{source/puits, pas de circulation}}.$$
Seule la **partie rotationnelle** correspond à une vraie fonction de courant. On veut donc l'extraire proprement et **jeter** le reste.

Pour cela on introduit la **vorticité** (le « taux de rotation local » de l'écoulement). En appliquant l'opérateur rotationnel à la définition de $\psi$, les deux relations $\rho_0 u=-\partial_z\psi$ et $\rho_0 w=+\partial_x\psi$ se combinent en **une seule équation** :
$$\boxed{\nabla^2\psi \;=\; \frac{\partial(\rho_0 w)}{\partial x} - \frac{\partial(\rho_0 u)}{\partial z}}$$
où $\nabla^2 = \partial_x^2 + \partial_z^2$ est le **Laplacien**. C'est une **équation de Poisson** : le membre de droite (calculable directement à partir des données $u,w$) est connu, et on cherche $\psi$.

**Comment on la résout numériquement ?**
1. On calcule le membre de droite (la vorticité de masse) par différences finies.
2. On écrit le Laplacien comme une grande **matrice creuse** (chaque point relié à ses 4 voisins).
3. On impose une **condition au bord** : ici $\psi=0$ sur le contour du domaine (Dirichlet). *(À adapter en périodique si le domaine l'est — voir remarque en fin de notebook.)*
4. On résout le système linéaire $A\,\psi = b$ en une fois (`scipy.sparse.linalg.spsolve`).

**Le résultat** est un $\psi$ **unique** et **indépendant du chemin** : la résolution projette automatiquement le champ sur sa partie rotationnelle et laisse de côté la partie divergente.

---
## Étape 6 — Et c'est ici que ça rejoint le sujet du stage

La part qu'on **jette** (la fraction divergente) n'est pas un déchet : c'est un **diagnostic**. Si une grosse partie de l'écoulement n'est *pas* capturée par $\psi$, c'est que l'écoulement s'éloigne d'un panache de masse bien organisé — un argument concret de plus contre l'idée qu'une fermeture « flux de masse » suffit à représenter le transport de quantité de mouvement.

Le code de la section **1bis** met tout ça en œuvre et renvoie justement cette fraction divergente. Les sections suivantes (4 à 9) confrontent ensuite cette circulation à la corrélation $\langle u'w'\rangle$.

---
### 🔑 À retenir en une phrase
> Une ligne de courant est une isoligne de la fonction de courant $\psi$ ; en atmosphère on travaille avec $\psi$ **de masse** ($\rho_0 u=-\partial_z\psi$, $\rho_0 w=+\partial_x\psi$) ; et pour l'obtenir proprement sur des données réelles on **résout une équation de Poisson** $\nabla^2\psi=\partial_x(\rho_0 w)-\partial_z(\rho_0 u)$ plutôt que d'intégrer naïvement.

## 0. Imports & configuration

Calé sur le pipeline **large300** : fichiers 3D séparés par variable dans `3D/MESONH_RCE_large300_3D_{var}.nc` (variables `ua`,`va`,`wa`,`ta`,`pa`,`hus`), altitude reprise du 1D, état stationnaire = **dernier tiers** des pas de temps, $\rho_0(z)$ calculé via la température virtuelle. Lecture **niveau par niveau** pour la RAM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import xarray as xr
import gc, os

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.25,
    'image.cmap': 'RdBu_r',
})

# ============================================================
#  CONFIGURATION  (identique au notebook rcemip_large300)
# ============================================================
DIR_3D = '3D';  DIR_2D = '2D';  DIR_1D = '1D'
def path3d(var): return os.path.join(DIR_3D, f'MESONH_RCE_large300_3D_{var}.nc')
def path1d(var): return os.path.join(DIR_1D, f'MESONH_RCE_large300_1D_{var}.nc')

Rd, Rv = 287.05, 461.5
EPSILON = Rd / Rv          # ≈ 0.622
BLOC    = 2                # taille de bloc temporel (RAM)

USE_SYNTHETIC = False      # True = champ jouet (démo sans données) ; False = large300
print('Config prête. USE_SYNTHETIC =', USE_SYNTHETIC)

## 1. Chargement large300 : coupe $(x,z)$ moyennée sur $y$ et sur le temps

On construit la coupe 2D $(z,x)$ en moyennant **sur $y$** (révèle l'overturning) **et sur le temps** (état stationnaire). En parallèle on accumule $\langle u'w'\rangle$ moyenné de la même façon, avec la **même convention d'anomalie spatiale** $(y,x)$ par niveau que ton pipeline.

Trois étapes : (1a) métadonnées + altitude, (1b) profil $\rho_0(z)$ via $T_v$, (1c) coupes moyennées niveau par niveau.

In [ ]:
# --- (1a) Métadonnées : dims, tailles, altitude, fenêtre stationnaire ---
if not USE_SYNTHETIC:
    _ds = xr.open_dataset(path3d('ua'));  _da = _ds['ua']
    dim_t, dim_z, dim_y, dim_x = _da.dims        # ordre (t, z, y, x)
    n_t = _da.sizes[dim_t];  n_z = _da.sizes[dim_z]
    n_y = _da.sizes[dim_y];  n_x = _da.sizes[dim_x]
    _ds.close();  del _ds, _da;  gc.collect()

    # altitude depuis le fichier 1D (plus fiable que la coord brute)
    _t1 = xr.open_dataset(path1d('ua_avg'))
    alt = _t1['altitude'].values.astype(float).copy()
    _t1.close()

    t_stat   = int(2 * n_t / 3)          # dernier tiers = stationnaire
    idx_stat = slice(t_stat, None)
    n_stat   = n_t - t_stat
    print(f'Grille : {n_t} t × {n_z} z × {n_y} y × {n_x} x')
    print(f'Altitude : {alt[0]:.0f} → {alt[-1]:.0f} m')
    print(f'Stationnaire : t={t_stat}→{n_t-1} ({n_stat} pas)')

In [ ]:
# --- (1b) Profil rho_0(z) via température virtuelle (gaz parfaits) ---
if not USE_SYNTHETIC:
    rho0_sum = np.zeros(n_z);  n_rho = 0
    ds_ta  = xr.open_dataset(path3d('ta'))
    ds_pa  = xr.open_dataset(path3d('pa'))
    ds_hus = xr.open_dataset(path3d('hus'))
    for t0 in range(t_stat, n_t, BLOC):
        t1 = min(t0 + BLOC, n_t);  sl = {dim_t: slice(t0, t1)}
        T_blk  = ds_ta['ta'].isel(sl).values
        p_blk  = ds_pa['pa'].isel(sl).values
        qv_blk = ds_hus['hus'].isel(sl).values
        Tv_blk  = T_blk * (1.0 + qv_blk/EPSILON) / (1.0 + qv_blk)
        rho_blk = p_blk / (Rd * Tv_blk)
        rho0_sum += rho_blk.mean(axis=(0, 2, 3)) * (t1 - t0)
        n_rho    += (t1 - t0)
        del T_blk, p_blk, qv_blk, Tv_blk, rho_blk;  gc.collect()
    ds_ta.close();  ds_pa.close();  ds_hus.close()
    del ds_ta, ds_pa, ds_hus;  gc.collect()
    rho0 = rho0_sum / n_rho
    print(f'ρ₀ : surface {rho0[0]:.3f} → sommet {rho0[-1]:.4f} kg/m³ '
          f'({n_rho} pas)')

In [ ]:
# --- (1c) Coupes (z,x) moyennées sur y ET sur le temps stationnaire ---
#   U, W      : <.>_{y,t}                      -> pour psi (lignes de courant)
#   UPWP      : <u'w'>_{y,t}  (anomalie (y,x)) -> flux de Reynolds local (z,x)
#   ubar,wbar : <.>_{x,y,t}                    -> profils pour diagnostics
if not USE_SYNTHETIC:
    z = alt.copy()
    x = np.arange(n_x, dtype=float)            # indices x (remplace par dx réel si dispo)
    U  = np.zeros((n_z, n_x));  W = np.zeros((n_z, n_x))
    UPWP = np.zeros((n_z, n_x))
    ubar = np.zeros(n_z);  wbar = np.zeros(n_z)

    ds_u = xr.open_dataset(path3d('ua'))
    ds_w = xr.open_dataset(path3d('wa'))
    for iz in range(n_z):
        sl = {dim_t: idx_stat, dim_z: iz}
        u_lev = ds_u['ua'].isel(sl).values     # (n_stat, ny, nx)
        w_lev = ds_w['wa'].isel(sl).values
        # anomalie spatiale (y,x) par pas de temps -> convention pipeline
        u_p = u_lev - u_lev.mean(axis=(1, 2), keepdims=True)
        w_p = w_lev - w_lev.mean(axis=(1, 2), keepdims=True)
        # moyennes <.>_{y,t} -> profils en x
        U[iz]    = u_lev.mean(axis=(0, 1))      # moyenne sur t puis y
        W[iz]    = w_lev.mean(axis=(0, 1))
        UPWP[iz] = (u_p * w_p).mean(axis=(0, 1))
        ubar[iz] = u_lev.mean();  wbar[iz] = w_lev.mean()
        del u_lev, w_lev, u_p, w_p;  gc.collect()
        if iz % 10 == 0: print(f'  niveau {iz}/{n_z-1} ({alt[iz]:.0f} m)')
    ds_u.close();  ds_w.close();  del ds_u, ds_w;  gc.collect()

    upwp = UPWP                               # alias attendu par la suite
    print('Coupes (z,x) prêtes :', U.shape)

In [ ]:
# --- Repli synthétique (uniquement si USE_SYNTHETIC=True) : champ jouet ---
#     overturning sinusoïdal + panaches + bruit. Sert à exécuter le notebook
#     sans les NetCDF. NE PAS interpréter physiquement.
if USE_SYNTHETIC:
    rng = np.random.default_rng(0)
    n_z, n_x = 74, 256
    z = np.linspace(0, 20000, n_z);  x = np.linspace(0, 300000, n_x)
    XX, ZZ = np.meshgrid(x, z);  rho0 = 1.2*np.exp(-z/8000.)
    H, L = z.max(), x.max()
    psi0 = 4e3*np.sin(np.pi*ZZ/H)*np.sin(2*np.pi*XX/L)
    dz = z[1]-z[0];  dx = x[1]-x[0]
    W = np.gradient(psi0, dx, axis=1)/rho0[:,None]
    U = -np.gradient(psi0, dz, axis=0)/rho0[:,None]
    for xc in rng.uniform(0, L, 6):
        W += rng.uniform(3,8)*np.exp(-((XX-xc)/6000.)**2)*np.sin(np.pi*ZZ/H)
    U += 0.5*rng.standard_normal(U.shape);  W += 0.3*rng.standard_normal(W.shape)
    ubar = U.mean(axis=1);  wbar = W.mean(axis=1)
    upwp = (U-ubar[:,None])*(W-wbar[:,None])
    print('⚠️ MODE SYNTHÉTIQUE — données fabriquées, non physiques.')
    print('Grille (nz,nx) =', U.shape)

## 1bis. Fonction de courant de masse $\psi$ — résolution propre par Poisson

**Chaîne logique.** En anélastique, la continuité sur le champ moyenné en $y$ s'écrit
$$\partial_x(\rho_0\bar u) + \partial_z(\rho_0\bar w) = 0,$$
c.-à-d. le couple flux de masse $(\rho_0\bar u,\,\rho_0\bar w)$ est à **divergence nulle**. Cette nullité garantit l'existence d'une fonction de courant $\psi$ telle que
$$\rho_0\,\bar u = -\partial_z\psi,\qquad \rho_0\,\bar w = +\partial_x\psi.$$
Les **isolignes de $\psi$ sont les lignes de courant** (le flux de masse leur est tangent).

**Pourquoi pas une simple intégration ?** Le champ réel (moyenné sur un $y$ fini, bruité, discret) n'est *jamais* exactement non-divergent. Intégrer $\rho_0\bar w$ en $x$ ou $-\rho_0\bar u$ en $z$ donne alors des résultats **dépendants du chemin**. Il faut extraire proprement la seule partie qui porte une circulation.

**Décomposition de Helmholtz + Poisson.** On scinde le flux de masse en partie rotationnelle (la circulation) + partie divergente (le résidu, qu'on écarte). En prenant le rotationnel de la définition de $\psi$ on obtient une **équation de Poisson** :
$$\nabla^2\psi = \partial_x(\rho_0\bar w) - \partial_z(\rho_0\bar u) \;\equiv\; \omega_{\text{masse}},$$
avec $\psi=0$ au bord (Dirichlet). Sa solution est **unique, indépendante du chemin**, et projette automatiquement hors de la partie divergente.

👉 C'est le geste qui *tourne la page du flux de masse* : on ne **postule** plus que l'écoulement est un panache de masse organisé — on **extrait** la circulation réellement présente, et tout ce qui n'en relève pas (fraction divergente, structures non-rotationnelles) devient un diagnostic à part entière.

In [ ]:
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import spsolve

def mass_streamfunction(U, W, x, z, rho0):
    """Fonction de courant de masse par résolution de Poisson (Dirichlet).
    Résout  nabla^2 psi = d_x(rho0 W) - d_z(rho0 U)  avec psi=0 au bord.
    psi est unique et indépendant du chemin d'intégration ; la partie
    divergente du flux de masse est automatiquement projetée hors de psi.
    Renvoie (psi, frac_div) où frac_div = fraction divergente écartée."""
    nz, nx = U.shape
    dx = float(x[1]-x[0]);  dz = float(z[1]-z[0])
    rhoU = rho0[:, None] * U
    rhoW = rho0[:, None] * W
    # vorticité de masse = terme source du Poisson
    omega = (np.gradient(rhoW, dx, axis=1) - np.gradient(rhoU, dz, axis=0))
    # Laplacien 2D sur les noeuds intérieurs (Dirichlet psi=0 au bord)
    Ix = identity(nx-2);  Iz = identity(nz-2)
    Lx = diags([1, -2, 1], [-1, 0, 1], shape=(nx-2, nx-2)) / dx**2
    Lz = diags([1, -2, 1], [-1, 0, 1], shape=(nz-2, nz-2)) / dz**2
    A = (kron(Iz, Lx) + kron(Lz, Ix)).tocsr()
    psi = np.zeros((nz, nx))
    psi[1:-1, 1:-1] = spsolve(A, omega[1:-1, 1:-1].ravel()).reshape(nz-2, nx-2)
    # diagnostic : part divergente écartée (champ rotationnel reconstruit vs réel)
    rhoU_rot = -np.gradient(psi, dz, axis=0)
    rhoW_rot =  np.gradient(psi, dx, axis=1)
    frac_div = ((np.linalg.norm(rhoU-rhoU_rot)+np.linalg.norm(rhoW-rhoW_rot))
                / (np.linalg.norm(rhoU)+np.linalg.norm(rhoW)+1e-30))
    return psi, frac_div

psi, frac_div = mass_streamfunction(U, W, x, z, rho0)
print(f'Fraction divergente écartée (non capturée par ψ) : {frac_div:.1%}')
print('  → grande = écoulement loin d\'un pur flux de masse rotationnel '
      '(argument direct contre la fermeture mass-flux)')

## 2-3. Visualisation : lignes de courant + faible corrélation $u'w'$

On superpose :
- les **isolignes de $\psi$** (lignes de courant de masse) ;
- en fond couleur, le **flux de Reynolds local** $u'w'$.

👉 Le point clé pour répondre à *« comment voit-on la faible corrélation ? »* : la cellule d'overturning (lignes de courant bien organisées) coexiste avec un champ $u'w'$ qui **change de signe** / se compense spatialement. Visuellement l'organisation est nette, mais l'intégrale (la corrélation moyenne) reste faible car les contributions $+$ et $-$ s'annulent.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
Xkm, Zkm = x/1000, z/1000

# --- (a) lignes de courant + vitesse verticale
ax = axes[0]
pc = ax.pcolormesh(Xkm, Zkm, W, shading='auto',
                   norm=TwoSlopeNorm(vcenter=0), cmap='RdBu_r')
cs = ax.contour(Xkm, Zkm, psi, levels=14, colors='k', linewidths=0.7)
ax.set_title("(a) Lignes de courant de masse (ψ) + vitesse verticale w")
ax.set_ylabel('z [km]')
fig.colorbar(pc, ax=ax, label='w [m/s]', pad=0.01)

# --- (b) lignes de courant + flux de Reynolds u'w'
ax = axes[1]
vmax = np.percentile(np.abs(upwp), 98)
pc = ax.pcolormesh(Xkm, Zkm, upwp, shading='auto',
                   norm=TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax), cmap='RdBu_r')
ax.contour(Xkm, Zkm, psi, levels=14, colors='k', linewidths=0.7)
ax.set_title("(b) Mêmes lignes de courant + flux de Reynolds local u'w'")
ax.set_xlabel('x [km]');  ax.set_ylabel('z [km]')
fig.colorbar(pc, ax=ax, label="u'w' [m²/s²]", pad=0.01)
plt.tight_layout();  plt.show()

## 4. Que tirer de la visualisation ? — diagnostics quantitatifs

On quantifie l'intuition « organisation forte / corrélation faible » :
- **Coefficient de corrélation** $r(z) = \dfrac{\langle u'w'\rangle}{\sigma_u\,\sigma_w}$ par niveau : si $|r|\ll 1$, le transport net est faible *malgré* des $u',w'$ d'amplitude marquée.
- **Taux d'annulation spatiale** : $1 - \dfrac{|\int u'w'\,dx|}{\int |u'w'|\,dx}$ par niveau. Proche de 1 = les contributions se compensent spatialement (signature visuelle de la « faible corrélation »).

In [ ]:
# r(z) de la STRUCTURE ORGANISÉE (champ moyenné en y) : corrélation entre
# les ondulations en x de U et W. C'est la corrélation 'visible' à grande
# échelle — à comparer au vrai <u'w'> sous-maille (UPWP, calculé en 1c).
Up = U - U.mean(axis=1, keepdims=True)
Wp = W - W.mean(axis=1, keepdims=True)
cov  = (Up*Wp).mean(axis=1)
su   = Up.std(axis=1);  sw = Wp.std(axis=1)
r_z  = cov/(su*sw + 1e-30)

cancel = 1 - np.abs(upwp.sum(axis=1))/(np.abs(upwp).sum(axis=1)+1e-30)

fig, ax = plt.subplots(1, 2, figsize=(10,5), sharey=True)
ax[0].plot(r_z, Zkm, 'o-', ms=3)
ax[0].axvline(0, color='k', lw=0.8)
ax[0].set_xlabel("r(z) = ⟨u'w'⟩ / (σu σw)");  ax[0].set_ylabel('z [km]')
ax[0].set_title('Corrélation u\u2032-w\u2032 par niveau')
ax[0].set_xlim(-1, 1)

ax[1].plot(cancel, Zkm, 's-', ms=3, color='C3')
ax[1].set_xlabel('Taux d\'annulation spatiale')
ax[1].set_title('Compensation spatiale de u\u2032w\u2032')
ax[1].set_xlim(0, 1)
plt.tight_layout();  plt.show()

print(f"|r| médian = {np.median(np.abs(r_z)):.3f}  "
      f"(faible corrélation confirmée si ≪ 1)")

## 5. Lien avec la paramétrisation du flux de Reynolds

Rappel de ton résultat : $\Phi_{rey}=T_1+T_2$, $T_1$ = transport par flux de masse organisé, $T_2$ = covariance sous-maille (≈44 %, $r\approx+0.99$ avec $\Phi_{rey}$).

**Ce que les lignes de courant ajoutent** : elles montrent *où* dans le plan $(x,z)$ le transport organisé ($T_1$) est aligné avec la cellule, et où $u'w'$ s'écarte du cadre flux-de-masse. On peut superposer le **vecteur flux de masse** $(\rho_0\bar u,\rho_0\bar w)$ et regarder l'angle avec le **gradient** de $\bar u$ : une fermeture down-gradient suppose $\overline{u'w'}\propto-\partial_z\bar u$. Tracer cet angle révèle les zones de **contre-gradient** (up-gradient) — typiques du transport convectif organisé que le mélange turbulent ne capture pas.

In [ ]:
ubar_z = U.mean(axis=1)
dudz = np.gradient(ubar_z, z)
# flux moyen par niveau
flux_z = (Up*Wp).mean(axis=1)
# diffusivité turbulente implicite K = -flux/(du/dz) ; K<0 => contre-gradient
K = -flux_z/(dudz + 1e-30)

fig, ax = plt.subplots(figsize=(6,5))
ax.plot(K, Zkm, 'o-', ms=3)
ax.axvline(0, color='k', lw=0.8)
ax.fill_betweenx(Zkm, K, 0, where=(K<0), color='C3', alpha=0.3,
                 label='contre-gradient (K<0)')
ax.set_xlabel('K(z) = -⟨u\u2032w\u2032⟩ / (∂z ū)  [m²/s]')
ax.set_ylabel('z [km]');  ax.legend()
ax.set_title('Diffusivité implicite : zones non capturables par down-gradient')
plt.tight_layout();  plt.show()

## 6. Décomposition en échelles — Fourier & ondelettes

Objectif : **quelles échelles horizontales portent le transport $u'w'$ ?** Si le co-spectre $\widehat{u'w'}(k)$ est dominé par les grandes échelles (cellule), une approche flux-de-masse a une chance ; s'il est large-bande, il faut une approche multi-échelle.

Tu maîtrises déjà les ondelettes (DWT, OMP/OLS) — ici on commence par le **co-spectre de Fourier** (rapide, interprétable), puis une **ondelette continue (CWT)** pour localiser spatialement les structures porteuses de flux.

In [ ]:
# --- Co-spectre de Fourier de u'w' à un niveau donné
k_lvl = np.argmin(np.abs(Zkm - 5.0))   # ~5 km
up = Up[k_lvl];  wp = Wp[k_lvl]
n = len(x);  dx = x[1]-x[0]
Uf = np.fft.rfft(up);  Wf = np.fft.rfft(wp)
kx = np.fft.rfftfreq(n, d=dx)
cospec = (Uf*np.conj(Wf)).real / n        # co-spectre (partie réelle)
wavelength_km = np.where(kx>0, 1/kx/1000, np.nan)

fig, ax = plt.subplots(figsize=(7,4))
ax.semilogx(wavelength_km[1:], cospec[1:], 'o-', ms=3)
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('Longueur d\'onde [km]');  ax.set_ylabel("Co-spectre u'w'")
ax.set_title(f"Co-spectre du flux à z≈{Zkm[k_lvl]:.1f} km")
ax.invert_xaxis()
plt.tight_layout();  plt.show()

print('Échelle dominante du flux : '
      f'{wavelength_km[1:][np.argmax(np.abs(cospec[1:]))]:.0f} km')

In [ ]:
# --- CWT (ondelette de Morlet) pour localiser les structures du flux
# pip install pywavelets si besoin
try:
    import pywt
    scales = np.geomspace(2, n//4, 60)
    cwt_u, freqs = pywt.cwt(up, scales, 'morl', sampling_period=dx)
    cwt_w, _     = pywt.cwt(wp, scales, 'morl', sampling_period=dx)
    cocwt = (cwt_u*cwt_w)              # co-scalogramme (flux localisé en x et échelle)
    wl_km = 1/freqs/1000

    fig, ax = plt.subplots(figsize=(11,4))
    vmax = np.percentile(np.abs(cocwt), 98)
    pc = ax.pcolormesh(x/1000, wl_km, cocwt, shading='auto',
                       norm=TwoSlopeNorm(vcenter=0,vmin=-vmax,vmax=vmax), cmap='RdBu_r')
    ax.set_yscale('log');  ax.set_ylabel('Longueur d\'onde [km]')
    ax.set_xlabel('x [km]')
    ax.set_title(f"Co-scalogramme u'w' (où ET à quelle échelle le flux est transporté) — z≈{Zkm[k_lvl]:.1f} km")
    fig.colorbar(pc, ax=ax, label="contribution locale à u'w'")
    plt.tight_layout();  plt.show()
except ImportError:
    print('pywt non installé : pip install PyWavelets')

## 7. Extension aux lignes de courant 3D

En 3D la fonction de courant scalaire n'existe plus (il faudrait un vecteur). On visualise donc directement les **trajectoires d'intégration** du champ $(u,v,w)$ — *streamlines 3D* — à un instant. Utile pour voir les **rouleaux / cellules** de l'auto-agrégation.

Approche légère : intégrateur RK4 sur le champ interpolé, quelques graines, rendu 3D matplotlib. (Pour de gros volumes, préférer `pyvista`/ParaView.)

In [ ]:
from scipy.interpolate import RegularGridInterpolator

def streamlines_3d(U3, V3, W3, x, y, z, seeds, n_steps=400, dt=20.):
    """Intègre des lignes de courant 3D par RK4. U3,V3,W3 de forme (nz,ny,nx)."""
    pts = (z, y, x)
    fu = RegularGridInterpolator(pts, U3, bounds_error=False, fill_value=0)
    fv = RegularGridInterpolator(pts, V3, bounds_error=False, fill_value=0)
    fw = RegularGridInterpolator(pts, W3, bounds_error=False, fill_value=0)
    def vel(p):
        q = p[:, [2,1,0]]  # -> (z,y,x)
        return np.stack([fu(q), fv(q), fw(q)], axis=1)
    lines = []
    for s in seeds:
        p = np.array(s, float)[None,:];  traj=[p[0].copy()]
        for _ in range(n_steps):
            k1=vel(p); k2=vel(p+0.5*dt*k1); k3=vel(p+0.5*dt*k2); k4=vel(p+dt*k3)
            p = p + dt/6*(k1+2*k2+2*k3+k4)
            traj.append(p[0].copy())
        lines.append(np.array(traj))
    return lines

# Démo 3D synthétique (petit volume)
nz3,ny3,nx3 = 30,40,40
zc=np.linspace(0,2e4,nz3); yc=np.linspace(0,1e5,ny3); xc=np.linspace(0,1e5,nx3)
Zc,Yc,Xc=np.meshgrid(zc,yc,xc,indexing='ij')
U3= np.sin(np.pi*Zc/2e4)*np.cos(2*np.pi*Xc/1e5)
V3= 0.3*np.sin(2*np.pi*Yc/1e5)
W3= np.sin(np.pi*Zc/2e4)*np.sin(2*np.pi*Xc/1e5)
seeds=[(xs,5e4,1e3) for xs in np.linspace(1e4,9e4,8)]
lines=streamlines_3d(U3,V3,W3,xc,yc,zc,seeds)

fig=plt.figure(figsize=(8,6)); ax=fig.add_subplot(111,projection='3d')
for L in lines:
    ax.plot(L[:,0]/1e3, L[:,1]/1e3, L[:,2]/1e3, lw=1)
ax.set_xlabel('x [km]'); ax.set_ylabel('y [km]'); ax.set_zlabel('z [km]')
ax.set_title('Lignes de courant 3D (démo)')
plt.tight_layout(); plt.show()

## 8. Isolation des tourbillons — critère $\lambda_2$ / Okubo-Weiss

Pour isoler les tourbillons dans la coupe $(x,z)$, on utilise le **critère $Q$ / Okubo-Weiss 2D** : un point est dans un cœur tourbillonnaire si la rotation domine la déformation, i.e. 
$$W_{OW} = s_n^2 + s_s^2 - \omega^2 < 0,$$
avec $s_n=\partial_x u-\partial_z w$ (étirement normal), $s_s=\partial_z u+\partial_x w$ (cisaillement), $\omega=\partial_x w-\partial_z u$ (vorticité). Les zones $W_{OW}<0$ marquent les tourbillons.

In [ ]:
dudx=np.gradient(U,x,axis=1); dudz=np.gradient(U,z,axis=0)
dwdx=np.gradient(W,x,axis=1); dwdz=np.gradient(W,z,axis=0)
sn=dudx-dwdz; ss=dudz+dwdx; omega=dwdx-dudz
OW=sn**2+ss**2-omega**2
thr=-0.2*np.std(OW)

fig,ax=plt.subplots(figsize=(11,4))
pc=ax.pcolormesh(Xkm,Zkm,omega,shading='auto',
                 norm=TwoSlopeNorm(vcenter=0),cmap='RdBu_r')
ax.contour(Xkm,Zkm,OW,levels=[thr],colors='lime',linewidths=1.5)
ax.contour(Xkm,Zkm,psi,levels=12,colors='k',linewidths=0.5,alpha=0.5)
ax.set_xlabel('x [km]');ax.set_ylabel('z [km]')
ax.set_title('Vorticité ω + cœurs tourbillonnaires (Okubo-Weiss<0, vert) + ψ')
fig.colorbar(pc,ax=ax,label='ω [1/s]')
plt.tight_layout();plt.show()

## 9. Autres idées (ajouts pertinents pour le stage)

Quelques diagnostics complémentaires qui s'inscrivent dans l'objectif *« comprendre pourquoi $u'w'$ échappe au flux de masse »* :

1. **Analyse en quadrants** (Q1–Q4) du flux $u'w'$ : décompose le transport en éjections ($u'<0,w'>0$) vs balayages ($u'>0,w'<0$). Permet de voir si le flux net faible vient d'un quasi-équilibre entre quadrants — signature directe de la « faible corrélation ».
2. **Angle de phase $u'$–$w'$** par niveau : un déphasage proche de $90°$ explique une covariance nulle malgré des amplitudes fortes.
3. **Moyenne temporelle** des lignes de courant sur plusieurs $t$ : isole la circulation *stationnaire* (overturning d'agrégation) du transitoire turbulent.
4. **Conditionnement par cœur tourbillonnaire** : recalculer $T_1$/$T_2$ *dans* vs *hors* tourbillons (réutilise ton framework up/dn/env).

Ci-dessous : l'analyse en quadrants, la plus directement reliée à ta problématique.

In [ ]:
# Analyse en quadrants au niveau k_lvl
up=Up[k_lvl]; wp=Wp[k_lvl]; fl=up*wp
Q={'Q1 (u\u2032>0,w\u2032>0)':(up>0)&(wp>0),
   'Q2 éjection (u\u2032<0,w\u2032>0)':(up<0)&(wp>0),
   'Q3 (u\u2032<0,w\u2032<0)':(up<0)&(wp<0),
   'Q4 balayage (u\u2032>0,w\u2032<0)':(up>0)&(wp<0)}
contrib={kk:fl[m].sum() for kk,m in Q.items()}

fig,ax=plt.subplots(figsize=(7,4))
ax.bar(range(4),list(contrib.values()),
       color=['C0','C2','C0','C3'])
ax.set_xticks(range(4)); ax.set_xticklabels(list(contrib.keys()),rotation=15,ha='right',fontsize=9)
ax.axhline(0,color='k',lw=0.8)
ax.set_ylabel("Contribution à Σ u'w'")
ax.set_title(f'Analyse en quadrants — z≈{Zkm[k_lvl]:.1f} km')
plt.tight_layout();plt.show()

net=sum(contrib.values()); gross=sum(abs(v) for v in contrib.values())
print(f'Flux net / flux brut = {net/gross:+.2%}  '
      '(proche de 0 => forte compensation entre quadrants)')

---
## Synthèse / prochaines étapes

| Section | Ce qu'on regarde | Ce que ça dit sur la fermeture |
|---|---|---|
| 1bis-3 | Lignes de courant ψ + cartes w, u'w' | L'overturning est organisé mais u'w' change de signe |
| 4 | r(z), taux d'annulation | Faible corrélation = compensation spatiale, pas absence de structure |
| 5 | K(z) implicite | Zones de contre-gradient → down-gradient inadapté |
| 6 | Co-spectre / co-scalogramme | Échelles porteuses du flux (mass-flux viable ou non) |
| 7-8 | Streamlines 3D, tourbillons | Structures cohérentes candidates à un nouveau closure |
| 9 | Quadrants, phase | Mécanisme de la faible covariance |

**Données** : mettre `USE_SYNTHETIC=False` (section 0) et placer les NetCDF dans `3D/`, `2D/`, `1D/` selon l'arborescence `MESONH_RCE_large300_*`. La section 1 (1a–1c) charge alors les vraies coupes. `USE_SYNTHETIC=True` rebascule sur le champ jouet pour tester le code hors données.